In [9]:
import torch
import torch.nn as nn
import os

print("=" * 50)
print("ENVIRONMENT INFO")
print("=" * 50)

print("\nPyTorch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device tersedia:", device)
print("Jumlah GPU:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("✓ GPU aktif! Training akan menggunakan GPU.")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("⚠ GPU tidak terdeteksi")
    print("Jika di RunPod, pastikan pod sudah dijalankan dengan GPU")

# Check CUDA
print("\nCUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)

# Memory info
print("\n" + "=" * 50)

ENVIRONMENT INFO

PyTorch version: 2.4.1+cu124
Device tersedia: cuda
Jumlah GPU: 1
✓ GPU aktif! Training akan menggunakan GPU.
  GPU 0: NVIDIA RTX 2000 Ada Generation

CUDA Available: True
CUDA Version: 12.4



In [10]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

print(PROJECT_DIR)

/workspace/sibi-project/notebook


In [12]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/workspace/sibi-project")
RAW_DIR = PROJECT_DIR / "data" / "raw" / "sibi"

print("Project folder:", PROJECT_DIR)
print("Dataset folder:", RAW_DIR)

if RAW_DIR.exists():
    print("Folder dataset ditemukan.")
else:
    print("Folder dataset TIDAK ditemukan. Cek lagi struktur folder.")

Project folder: /workspace/sibi-project
Dataset folder: /workspace/sibi-project/data/raw/sibi
Folder dataset ditemukan.


In [7]:
from pathlib import Path
import pandas as pd
import os

# Deteksi environment - lokal atau RunPod
if os.path.exists("/workspace"):
    # Running di RunPod
    PROJECT_DIR = Path("/workspace/sibi-project")
    print("🚀 Running di RunPod")
else:
    # Running lokal
    PROJECT_DIR = Path("/Users/dickyaditya04/Downloads/AI")
    print("💻 Running lokal")

RAW_DIR = PROJECT_DIR / "data" / "raw" / "sibi"

print("\nProject folder:", PROJECT_DIR)
print("Dataset folder:", RAW_DIR)
print("PROJECT_DIR exists:", PROJECT_DIR.exists())
print("RAW_DIR exists:", RAW_DIR.exists())

if RAW_DIR.exists():
    print("✓ Folder dataset ditemukan.")
    # Tampilkan jumlah folder kelas
    sibi_classes = sorted([p.name for p in RAW_DIR.iterdir() if p.is_dir()])
    print(f"Kelas yang ditemukan: {sibi_classes}")
else:
    print("✗ Folder dataset TIDAK ditemukan.")
    print("Pastikan struktur folder sudah benar:")
    print(f"  {PROJECT_DIR}/data/raw/sibi/")
    print("\nStruktur folder saat ini:")
    if PROJECT_DIR.exists():
        for item in sorted(PROJECT_DIR.iterdir()):
            print(f"  {item.name}")

🚀 Running di RunPod

Project folder: /workspace/sibi-project
Dataset folder: /workspace/sibi-project/data/raw/sibi
PROJECT_DIR exists: True
RAW_DIR exists: True
✓ Folder dataset ditemukan.
Kelas yang ditemukan: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']


In [8]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class_dirs = sorted([p for p in RAW_DIR.iterdir() if p.is_dir()])

data_info = []

for cls_dir in class_dirs:
    images = [p for p in cls_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
    data_info.append({
        "kelas": cls_dir.name,
        "jumlah_gambar": len(images)
    })

df_info = pd.DataFrame(data_info)

display(df_info)

print("Total kelas :", len(df_info))
print("Total gambar:", df_info["jumlah_gambar"].sum())

,kelas,jumlah_gambar
0,A,539
1,B,541
2,C,387
3,D,379
4,E,498
5,F,420
6,G,345
7,H,364
8,I,360
9,K,319


Total kelas : 24
Total gambar: 8442


In [ ]:
import random
import shutil

# Folder hasil split
SPLIT_DIR = PROJECT_DIR / "data" / "processed" / "sibi_split"

TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR   = SPLIT_DIR / "val"
TEST_DIR  = SPLIT_DIR / "test"

# Rasio pembagian dataset
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

SEED = 42
random.seed(SEED)

print("Folder hasil split akan dibuat di:")
print(SPLIT_DIR)

In [ ]:
def split_dataset(raw_dir, split_dir):
    # Cek apakah folder split sudah ada
    if split_dir.exists():
        print("Folder split sudah ada.")
        print("Lokasi:", split_dir)
        print("Proses split dilewati agar data lama tidak tertimpa.")
        return

    print("Memulai proses split dataset...")

    split_dir.mkdir(parents=True, exist_ok=True)

    class_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])

    for cls_dir in class_dirs:
        class_name = cls_dir.name

        images = [
            p for p in cls_dir.iterdir()
            if p.suffix.lower() in IMAGE_EXTS
        ]

        random.shuffle(images)

        total_data = len(images)

        train_count = int(total_data * TRAIN_RATIO)
        val_count = int(total_data * VAL_RATIO)

        train_files = images[:train_count]
        val_files = images[train_count:train_count + val_count]
        test_files = images[train_count + val_count:]

        split_data = {
            "train": train_files,
            "val": val_files,
            "test": test_files
        }

        for split_name, file_list in split_data.items():
            target_folder = split_dir / split_name / class_name
            target_folder.mkdir(parents=True, exist_ok=True)

            for file_path in file_list:
                shutil.copy2(file_path, target_folder / file_path.name)

        print(
            f"Kelas {class_name}: "
            f"Total={total_data}, "
            f"Train={len(train_files)}, "
            f"Val={len(val_files)}, "
            f"Test={len(test_files)}"
        )

    print("\nSplit dataset selesai.")

split_dataset(RAW_DIR, SPLIT_DIR)

In [ ]:
def cek_folder_split(split_dir):
    hasil = []

    for split_name in ["train", "val", "test"]:
        split_path = split_dir / split_name

        if not split_path.exists():
            print(f"Folder {split_name} belum ada.")
            continue

        class_dirs = sorted([p for p in split_path.iterdir() if p.is_dir()])

        for cls_dir in class_dirs:
            images = [
                p for p in cls_dir.iterdir()
                if p.suffix.lower() in IMAGE_EXTS
            ]

            hasil.append({
                "data": split_name,
                "kelas": cls_dir.name,
                "jumlah_gambar": len(images)
            })

    return pd.DataFrame(hasil)

df_split = cek_folder_split(SPLIT_DIR)
display(df_split)

print("Total gambar hasil split:", df_split["jumlah_gambar"].sum())

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Ukuran input EfficientNet-B0
IMG_SIZE = 224

# Batch size
BATCH_SIZE = 32

# Seed agar hasil tetap stabil
SEED = 42
torch.manual_seed(SEED)

# Data augmentation untuk training
train_transforms = transforms.Compose([
    transforms.RandomRotation(degrees=5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ColorJitter(contrast=0.1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Transforms untuk validation dan test (tanpa augmentation)
val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset = ImageFolder(VAL_DIR, transform=val_test_transforms)
test_dataset = ImageFolder(TEST_DIR, transform=val_test_transforms)

# Create dataloaders
train_ds = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_ds = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_ds = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Nama kelas:", class_names)
print("Jumlah kelas:", num_classes)
print(f"Total training samples: {len(train_dataset)}")
print(f"Total validation samples: {len(val_dataset)}")
print(f"Total test samples: {len(test_dataset)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualisasi batch pertama dari training set
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
axes = axes.flatten()

# Ambil satu batch
for batch_idx, (images, labels) in enumerate(train_ds):
    if batch_idx == 0:
        for i in range(12):
            # Denormalize untuk visualisasi
            img = images[i].numpy()
            img = np.transpose(img, (1, 2, 0))  # C, H, W -> H, W, C
            img = (img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
            img = np.clip(img, 0, 1)
            
            axes[i].imshow(img)
            label_idx = labels[i].item()
            axes[i].set_title(f"Label: {class_names[label_idx]}")
            axes[i].axis("off")
    break

plt.tight_layout()
plt.show()

In [ ]:
# PyTorch DataLoader sudah menghandle prefetching dengan num_workers
# Tidak perlu setup tambahan seperti di TensorFlow
print("✓ DataLoader sudah dioptimasi dengan num_workers=4")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Hitung class labels dari training dataset
train_labels = []
for _, labels in train_ds:
    train_labels.extend(labels.numpy())

train_labels = np.array(train_labels)

# Compute class weights untuk imbalanced dataset
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_labels
)

# Convert ke torch tensor
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

print("Class weights:")
for i, w in enumerate(weights):
    print(f"  {class_names[i]}: {w:.3f}")

In [ ]:
import torch.nn as nn
import torchvision.models as models

# Definisikan model EfficientNet-B0 dengan PyTorch
class EfficientNetB0SIBI(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super(EfficientNetB0SIBI, self).__init__()
        
        # Load pre-trained EfficientNet-B0
        # Menggunakan torchvision yang tersedia
        self.base_model = models.efficientnet_b0(pretrained=pretrained)
        
        # Freeze base model layers awalnya
        for param in self.base_model.parameters():
            param.requires_grad = False
        
        # Get input features dari classifier
        num_features = self.base_model.classifier[1].in_features
        
        # Replace classifier
        self.base_model.classifier = nn.Sequential(
            nn.Dropout(p=0.30, inplace=False),
            nn.Linear(num_features, num_classes)
        )
    
    def forward(self, x):
        return self.base_model(x)
    
    def unfreeze_for_finetuning(self, num_freeze_layers=50):
        """Unfreeze layers untuk fine-tuning"""
        # Unfreeze semua parameter
        for param in self.base_model.parameters():
            param.requires_grad = True
        
        # Freeze batch norm layers untuk stability
        for module in self.base_model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()
                for param in module.parameters():
                    param.requires_grad = False

# Create model
model = EfficientNetB0SIBI(num_classes=num_classes, pretrained=True)
model = model.to(device)

# Loss function dengan class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer - awalnya dengan learning rate 1e-3
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3
)

print("✓ Model EfficientNet-B0 PyTorch berhasil dibuat")
print(f"Total trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Device: {device}")

In [ ]:
# ===============================
# DEFINISI FOLDER OUTPUT
# ===============================

MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"
FEATURE_DIR = PROJECT_DIR / "features"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "best_model.keras"
FINAL_MODEL_PATH = MODEL_DIR / "final_model.keras"
CLASS_NAMES_PATH = MODEL_DIR / "class_names.json"

print("Folder model :", MODEL_DIR)
print("Folder report:", REPORT_DIR)
print("Folder fitur :", FEATURE_DIR)
print("Path model terbaik:", BEST_MODEL_PATH)

In [ ]:
# PyTorch Training Utils
class TrainingUtils:
    def __init__(self, best_model_path, patience=5):
        self.best_model_path = best_model_path
        self.best_val_acc = 0.0
        self.patience = patience
        self.patience_counter = 0
        self.best_weights = None
    
    def save_checkpoint(self, model, optimizer, epoch, val_acc, val_loss):
        """Simpan model jika val_acc lebih baik"""
        if val_acc > self.best_val_acc:
            self.best_val_acc = val_acc
            self.patience_counter = 0
            
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss
            }
            torch.save(checkpoint, str(self.best_model_path))
            print(f"✓ Model terbaik disimpan! Val Acc: {val_acc:.4f}")
            return True
        else:
            self.patience_counter += 1
            if self.patience_counter >= self.patience:
                return False  # Early stopping
            return True
    
    def load_best_model(self, model, optimizer):
        """Load model terbaik"""
        checkpoint = torch.load(str(self.best_model_path))
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        return model, optimizer

# Initialize training utils
train_utils = TrainingUtils(best_model_path=BEST_MODEL_PATH, patience=5)

print("✓ Training utilities siap")
print(f"Model terbaik akan disimpan di: {BEST_MODEL_PATH}")

In [ ]:
EPOCHS_STAGE_1 = 15

# Dictionary untuk menyimpan history
history_stage_1 = {
    'loss': [],
    'accuracy': [],
    'val_loss': [],
    'val_accuracy': []
}

print("=" * 60)
print("TRAINING STAGE 1: Transfer Learning (Base Model Frozen)")
print("=" * 60)

for epoch in range(EPOCHS_STAGE_1):
    # ========== TRAINING ==========
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_idx, (images, labels) in enumerate(train_ds):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch [{batch_idx + 1}/{len(train_ds)}] Loss: {loss.item():.4f}")
    
    train_loss /= len(train_ds)
    train_acc = train_correct / train_total
    
    # ========== VALIDATION ==========
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_ds:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)
    
    val_loss /= len(val_ds)
    val_acc = val_correct / val_total
    
    # Store history
    history_stage_1['loss'].append(train_loss)
    history_stage_1['accuracy'].append(train_acc)
    history_stage_1['val_loss'].append(val_loss)
    history_stage_1['val_accuracy'].append(val_acc)
    
    # Print epoch info
    print(f"Epoch [{epoch+1}/{EPOCHS_STAGE_1}]")
    print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
    
    # Save best model & early stopping check
    if not train_utils.save_checkpoint(model, optimizer, epoch, val_acc, val_loss):
        print("⚠ Early stopping - validation loss tidak meningkat")
        break

print("\n✓ Training Stage 1 selesai!")

In [ ]:
import subprocess

# Monitor GPU usage
print("\n" + "=" * 50)
print("GPU MONITORING")
print("=" * 50)

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except:
    print("nvidia-smi tidak tersedia")

In [ ]:
print("Training tahap 1 selesai.")

last_train_acc = history_stage_1['accuracy'][-1]
last_val_acc = history_stage_1['val_accuracy'][-1]
last_train_loss = history_stage_1['loss'][-1]
last_val_loss = history_stage_1['val_loss'][-1]

print("\nAkurasi training terakhir   :", round(last_train_acc * 100, 2), "%")
print("Akurasi validation terakhir :", round(last_val_acc * 100, 2), "%")
print("Loss training terakhir      :", round(last_train_loss, 4))
print("Loss validation terakhir    :", round(last_val_loss, 4))

In [ ]:
# ===============================
# FINE-TUNING SETUP
# ===============================

print("Setting up fine-tuning...")

# Unfreeze model untuk fine-tuning
model.unfreeze_for_finetuning(num_freeze_layers=50)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Create new optimizer dengan learning rate yang lebih kecil
optimizer_finetune = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-5
)

print("✓ Fine-tuning siap dengan learning rate: 1e-5")

In [ ]:
# ===============================
# SETUP UNTUK FINE-TUNING TRAINING
# ===============================

# Reset patience counter untuk fine-tuning
train_utils.patience_counter = 0

print("✓ Setup fine-tuning training selesai")

In [ ]:
EPOCHS_STAGE_2 = 10

# Dictionary untuk menyimpan history stage 2
history_stage_2 = {
    'loss': [],
    'accuracy': [],
    'val_loss': [],
    'val_accuracy': []
}

print("=" * 60)
print("TRAINING STAGE 2: Fine-Tuning (Selected Layers)")
print("=" * 60)

for epoch in range(EPOCHS_STAGE_2):
    # ========== TRAINING ==========
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_idx, (images, labels) in enumerate(train_ds):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer_finetune.zero_grad()
        loss.backward()
        optimizer_finetune.step()
        
        # Track metrics
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch [{batch_idx + 1}/{len(train_ds)}] Loss: {loss.item():.4f}")
    
    train_loss /= len(train_ds)
    train_acc = train_correct / train_total
    
    # ========== VALIDATION ==========
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_ds:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)
    
    val_loss /= len(val_ds)
    val_acc = val_correct / val_total
    
    # Store history
    history_stage_2['loss'].append(train_loss)
    history_stage_2['accuracy'].append(train_acc)
    history_stage_2['val_loss'].append(val_loss)
    history_stage_2['val_accuracy'].append(val_acc)
    
    # Print epoch info
    print(f"Epoch [{epoch+1}/{EPOCHS_STAGE_2}]")
    print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
    
    # Save best model & early stopping check
    if not train_utils.save_checkpoint(model, optimizer_finetune, epoch, val_acc, val_loss):
        print("⚠ Early stopping - validation loss tidak meningkat")
        break

print("\n✓ Fine-Tuning Stage 2 selesai!")

In [ ]:
print("Fine-tuning selesai.")

last_train_acc_2 = history_stage_2['accuracy'][-1]
last_val_acc_2 = history_stage_2['val_accuracy'][-1]
last_train_loss_2 = history_stage_2['loss'][-1]
last_val_loss_2 = history_stage_2['val_loss'][-1]

print("\nAkurasi training tahap 2   :", round(last_train_acc_2 * 100, 2), "%")
print("Akurasi validation tahap 2 :", round(last_val_acc_2 * 100, 2), "%")
print("Loss training tahap 2      :", round(last_train_loss_2, 4))
print("Loss validation tahap 2    :", round(last_val_loss_2, 4))

In [ ]:
# ===============================
# EVALUASI MODEL TERBAIK PADA DATA TEST
# ===============================

# Load best model
checkpoint = torch.load(str(BEST_MODEL_PATH))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Evaluate pada test set
test_loss = 0.0
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_ds:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        test_correct += (predicted == labels).sum().item()
        test_total += labels.size(0)

test_loss /= len(test_ds)
test_acc = test_correct / test_total

print("Hasil evaluasi data test:")
print("Test Loss    :", round(test_loss, 4))
print("Test Accuracy:", round(test_acc * 100, 2), "%")

In [ ]:
# ===============================
# CLASSIFICATION REPORT
# ===============================

from sklearn.metrics import classification_report
import numpy as np
import pandas as pd

y_true = []
y_pred = []

model.eval()
with torch.no_grad():
    for images, labels in test_ds:
        images = images.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        y_pred.extend(predicted.cpu().numpy())
        y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

df_report = pd.DataFrame(report_dict).transpose()
display(df_report)

# Simpan report
REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = REPORT_DIR / "classification_report.csv"
df_report.to_csv(REPORT_PATH)

print("Classification report disimpan di:")
print(REPORT_PATH)

In [ ]:
# ===============================
# CONFUSION MATRIX
# ===============================

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(14, 14))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(ax=ax, xticks_rotation=90)

plt.title("Confusion Matrix Klasifikasi Abjad SIBI")
plt.grid(False)

CM_PATH = REPORT_DIR / "confusion_matrix.png"
plt.savefig(CM_PATH, dpi=300, bbox_inches="tight")

plt.show()

print("Confusion matrix disimpan di:")
print(CM_PATH)

In [ ]:
# ===============================
# GRAFIK ACCURACY DAN LOSS
# ===============================

def combine_history_pytorch(h1, h2):
    """Combine history dari stage 1 dan stage 2"""
    history = {}
    for key in h1.keys():
        history[key] = h1[key] + h2.get(key, [])
    return history

history_all = combine_history_pytorch(history_stage_1, history_stage_2)

# Grafik Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history_all["accuracy"], label="Training Accuracy")
plt.plot(history_all["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Grafik Accuracy Training EfficientNet-B0 SIBI")
plt.legend()
plt.grid(True)

ACC_PATH = REPORT_DIR / "accuracy_curve.png"
plt.savefig(ACC_PATH, dpi=300, bbox_inches="tight")
plt.show()

# Grafik Loss
plt.figure(figsize=(8, 5))
plt.plot(history_all["loss"], label="Training Loss")
plt.plot(history_all["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Grafik Loss Training EfficientNet-B0 SIBI")
plt.legend()
plt.grid(True)

LOSS_PATH = REPORT_DIR / "loss_curve.png"
plt.savefig(LOSS_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Grafik accuracy disimpan di:", ACC_PATH)
print("Grafik loss disimpan di    :", LOSS_PATH)

In [ ]:
# ===============================
# SIMPAN MODEL FINAL DAN NAMA KELAS
# ===============================

import json

FINAL_MODEL_PATH = MODEL_DIR / "final_model.pt"
CLASS_NAMES_PATH = MODEL_DIR / "class_names.json"

# Simpan full checkpoint
final_checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'num_classes': num_classes
}

torch.save(final_checkpoint, str(FINAL_MODEL_PATH))

# Simpan class names juga dalam JSON
with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(class_names, f)

print("Model final disimpan di:")
print(FINAL_MODEL_PATH)

print("Nama kelas disimpan di:")
print(CLASS_NAMES_PATH)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import transforms
from PIL import Image

# Path model dan class names
FINAL_MODEL_PATH = MODEL_DIR / "final_model.pt"
CLASS_NAMES_PATH = MODEL_DIR / "class_names.json"

# Load checkpoint
checkpoint = torch.load(str(FINAL_MODEL_PATH))
class_names_loaded = checkpoint['class_names']
num_classes_loaded = checkpoint['num_classes']

# Create model dan load weights
test_model = EfficientNetB0SIBI(num_classes=num_classes_loaded, pretrained=False)
test_model.load_state_dict(checkpoint['model_state_dict'])
test_model = test_model.to(device)
test_model.eval()

print("✓ Model berhasil dimuat.")
print("Nama kelas:", class_names_loaded)

In [ ]:
from pathlib import Path
import os

# Untuk Jupyter lokal - pilih file gambar dari folder
print("Pilih file gambar SIBI untuk diprediksi")
print("\nGambar di folder saat ini:")

# Cari gambar di folder mana pun
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
all_images = []

# Search di berbagai folder
search_paths = [
    PROJECT_DIR / "data" / "raw" / "sibi",
    PROJECT_DIR / "data" / "processed",
    Path.home() / "Downloads",
    Path.cwd()
]

for search_path in search_paths:
    if search_path.exists():
        for img_file in search_path.rglob("*"):
            if img_file.suffix.lower() in image_exts:
                all_images.append(img_file)

if all_images:
    print(f"\nDitemukan {len(all_images)} gambar:")
    for i, img in enumerate(all_images[:10], 1):
        print(f"  {i}. {img}")
    
    # Gunakan gambar pertama atau ubah index sesuai keinginan
    image_path = str(all_images[0])
    print(f"\n✓ Menggunakan: {image_path}")
else:
    # Alternatif: input path manual
    print("\nTidak ada gambar yang ditemukan otomatis.")
    print("Silakan masukkan path gambar secara manual:")
    image_path = input("Path file gambar: ")
    
print("Gambar:", image_path)

In [ ]:
def predict_sibi_image(image_path, model, class_names, img_size=(224, 224)):
    # Transform untuk prediction
    predict_transform = transforms.Compose([
        transforms.Resize((img_size[0], img_size[1])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Load dan transform image
    img = Image.open(image_path).convert('RGB')
    img_tensor = predict_transform(img).unsqueeze(0).to(device)
    
    # Prediksi
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        prediction = probabilities[0].cpu().numpy()
    
    # Ambil prediksi tertinggi
    top_index = np.argmax(prediction)
    top_class = class_names[top_index]
    confidence = prediction[top_index]
    
    return top_class, confidence, prediction, img

# Jalankan prediksi
predicted_class, confidence, all_predictions, img = predict_sibi_image(
    image_path=image_path,
    model=test_model,
    class_names=class_names_loaded
)

print("===================================")
print("HASIL PREDIKSI")
print("===================================")
print("Gambar input       :", image_path)
print("Prediksi huruf SIBI:", predicted_class)
print("Confidence         :", round(confidence * 100, 2), "%")

# Tampilkan gambar
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis("off")
plt.title(f"Prediksi: {predicted_class} ({confidence * 100:.2f}%)")
plt.tight_layout()
plt.show()

In [ ]:
top_5_indices = np.argsort(all_predictions)[::-1][:5]

print("Top 5 Prediksi:")

for idx in top_5_indices:
    label = class_names_loaded[idx]
    score = all_predictions[idx] * 100
    print(f"{label}: {score:.2f}%")